# 02 · Translation logit lens (A1)

Reproduce Wendler et al. (2024) on Qwen3: for each concept, prompt `Français: "nuage" - 中文: "`, decode every
layer's residual stream at the last position with the `ln_final` lens, and compare P(target word), P(latent-language
word) and matched-vs-control words across layers.

Code lives in `src/latent_switch/`: `prompts` · `models` · `lens` · `measures` · `stats` · `translation` · `plotting`.
This notebook runs it and keeps the checks. Findings: `notes/lab_notebook.md` (2026-10-19).

In [ ]:
%load_ext autoreload
%autoreload 2

### Variables in this notebook

**Language / data**
- `SRC`, `TGT`, `LATENT`: language codes (FLORES style): source we translate *from*, target we translate *to*,
  and the candidate latent ("hub") language we look for in the middle layers. e.g. fra_Latn, cmn_Hans, eng_Latn.
- `ROLES`: role → language for the words we measure, e.g. {"tgt": TGT, "lat": LATENT}. Role names prefix curve names.
- `stats`: long table, one row per (concept, language): the word plus its token info for this tokenizer.
- `df`: wide table, one row per concept kept for this SRC→TGT pair. Columns per language:
  `{lang}` = the word, `{lang}_id` = its first token id (plain, no leading space),
  `{lang}_prefix` = list of ids of every token that is a prefix of the word (± leading space; TGT and LATENT only).
- `prompts`: list of strings, `prompts[i]` is the 5-shot prompt whose answer is `df.loc[i, TGT]`.
- `ctrl_idx`: array, `ctrl_idx[i]` = row of the concept used as the *control* for prompt i (never i itself;
  each concept used once).

**Model**
- `model`: the TransformerBridge model. `tok`: its tokenizer.
- `N_LAYERS`: number of blocks (28). `N_REAL`: real vocab size; `W_U` columns ≥ N_REAL are padding.

**Per prompt**
- `probs`: [N_LAYERS, d_vocab] logit-lens next-token probabilities at the last position, one row per layer.
- `m`: dict of curves for one prompt, each [N_LAYERS] (output of `measure`).

**Curve names (keys of `m` and `curves`)**
- `tgt_first` / `tgt_prefix`: P(target word) via its first token / via its prefix set.
- `lat_first` / `lat_prefix`: P(matching latent-language word), e.g. "cloud" for 云.
- `lat_ctrl_first` / `lat_ctrl_prefix`: P(the *control* concept's latent word) on the same prompt (overlapping ids removed).
- `tgt_ctrl_first` / `tgt_ctrl_prefix`: P(the *control* concept's target word) on the same prompt.
- `rank_tgt`, `rank_lat`, `rank_tgt_ctrl`, `rank_lat_ctrl`: rank of that first token among all tokens (0 = top).
- `entropy`: entropy of the layer's distribution, in bits.
- `script_{Latn,Hans,...}`: P(any token written in that script). Script ≠ language (Latin = en + fr + de ...).

**Script-level**
- `scripts`: script codes of SRC/TGT/LATENT. `masks`: script → bool [d_vocab] of that script's tokens.
- `correct`: bool array, top1 == target first token (word-level curves are plotted on these prompts).

**All prompts**
- `curves`: dict, same keys as `m`, each [n_prompts, N_LAYERS]; row i = prompt i.
- `top1`: list, the final layer's most likely token id per prompt.
- `correct`: bool array, top1 == target first token.

The naming pattern: tgt / lat / ctrl = which word, and first / prefix / rank = how we measure it.

In [ ]:
from pathlib import Path

import numpy as np
import torch
from transformers import AutoTokenizer

from latent_switch.concepts import add_token_info, drop_ambiguous, load_wendler, translation_set
from latent_switch.languages import script_of
from latent_switch.lens import layer_probs, script_mask
from latent_switch.measures import make_controls, measure
from latent_switch.models import load_model
from latent_switch.plotting import plot_paired, plot_run
from latent_switch.prompts import build_prompts
from latent_switch.translation import load_run, run_translation_lens, save_run

torch.set_grad_enabled(False)                      # inference only: no autograd graph

REPO = Path.cwd().parent
WENDLER = REPO / "external/llm-latent-language/data/langs"
OUT = REPO / "results" / "a1"
MODEL = "Qwen/Qwen3-0.6B-Base"

SRC, TGT, LATENT = "fra_Latn", "cmn_Hans", "eng_Latn"
ROLES = {"tgt": TGT, "lat": LATENT}

tok = AutoTokenizer.from_pretrained(MODEL)
stats = add_token_info(drop_ambiguous(load_wendler(WENDLER)), tok)
df = translation_set(stats, SRC, TGT, LATENT)
df.head()

## Step 1 · Prompts (`prompts.build_prompts`)

In [ ]:
prompts = build_prompts(df, SRC, TGT, k_shot=5, seed=0)
print(len(prompts), "prompts\n")
print(prompts[7])

In [ ]:
# Checks: reproducible, query is last, and the prompt's next token is the answer's plain first token
assert build_prompts(df, SRC, TGT) == prompts                       # same seed → same prompts
i = 8
print(repr(prompts[i][-40:]))
answer_ids = tok.encode(prompts[i] + df.at[i, TGT], add_special_tokens=False)
prompt_ids = tok.encode(prompts[i], add_special_tokens=False)
print("prompt is a token prefix of prompt+answer:", answer_ids[:len(prompt_ids)] == prompt_ids)
print("next token:", repr(tok.decode(answer_ids[len(prompt_ids)])), "| expected id",
      df.at[i, f"{TGT}_id"], "got", answer_ids[len(prompt_ids)])

## Step 2 · Per-layer probabilities (`lens.layer_probs`)

In [ ]:
model = load_model(MODEL)
N_LAYERS = model.cfg.n_layers                      # 28 for Qwen3-0.6B
N_REAL = len(tok)                                  # real tokens; W_U columns beyond this are padding
print(N_LAYERS, model.cfg.d_model, model.W_U.shape, N_REAL)

In [ ]:
i = 7                                                       # NUAGE → 云
probs, final_logits = layer_probs(model, prompts[i])
print("shape:", probs.shape)
print("rows sum to 1:", torch.allclose(probs.sum(-1), torch.ones(N_LAYERS, device=probs.device)))
print("padded vocab prob:", probs[:, N_REAL:].sum().item())

# The last lens row must equal the model's real output distribution
final_logits[N_REAL:] = -torch.inf
print("last layer == model output:", torch.allclose(probs[-1], final_logits.softmax(-1), atol=1e-5))

# Top-3 tokens at a few layers
for L in [0, 7, 14, 18, 21, 24, 27]:
    top = probs[L].topk(3)
    print(f"L{L:2d}", [(tok.decode(t), round(p, 3)) for p, t in zip(top.values.tolist(), top.indices.tolist())])

In [ ]:
row = df.loc[i]
en_set = torch.tensor(row[f"{LATENT}_prefix"])
zh_set = torch.tensor(row[f"{TGT}_prefix"])
en_id = row[f"{LATENT}_id"]

p_en = probs[:, en_set].sum(-1)                           # [28]
p_zh = probs[:, zh_set].sum(-1)                           # [28]
rank_en = (probs > probs[:, [en_id]]).sum(-1)             # [28]  0 = top token

print(row["concept_id"], [tok.decode(t) for t in row[f"{LATENT}_prefix"]])
for L in range(N_LAYERS):
    print(f"L{L:2d}  P(en)={p_en[L]:.4f}  P(zh)={p_zh[L]:.4f}  rank(en first token)={rank_en[L]}")

## Step 3 · Measures with controls (`measures.make_controls`, `measures.measure`)

In [ ]:
ctrl_idx = make_controls(len(df))
assert (ctrl_idx != np.arange(len(df))).all()             # nobody is their own control
assert sorted(ctrl_idx) == list(range(len(df)))           # each concept used exactly once
print(df.at[4, "concept_id"], "→ control:", df.at[ctrl_idx[4], "concept_id"])

In [ ]:
i = 7
m = measure(probs, df.loc[i], df.loc[ctrl_idx[i]], ROLES)
print(df.at[i, "concept_id"], "| control word:", df.at[ctrl_idx[i], LATENT])
print("all curves shape (28,):", all(v.shape == (N_LAYERS,) for v in m.values()))

for L in range(N_LAYERS):
    print(f"L{L:2d}  tgt={m['tgt_first'][L]:.3f}  lat={m['lat_first'][L]:.4f}  ctrl={m['lat_ctrl_first'][L]:.4f}"
          f"  | rank lat={int(m['rank_lat'][L]):6d}  ctrl={int(m['rank_lat_ctrl'][L]):6d}  | H={m['entropy'][L]:.2f}")

## Step 4 · All prompts, script masks, save (`lens.script_mask`, `translation.run_translation_lens`)

In [ ]:
scripts = sorted({script_of(code) for code in (SRC, TGT, LATENT)})                # e.g. ['Hans', 'Latn']
masks = {s: script_mask(tok, s, model.cfg.d_vocab).to(model.cfg.device) for s in scripts}
for s, mk in masks.items():
    ids = mk.nonzero().squeeze(-1)[::5000]
    print(s, int(mk.sum()), "tokens, e.g.", [tok.decode(t) for t in ids[:8].tolist()])

In [ ]:
run = run_translation_lens(model, df, prompts, ctrl_idx, ROLES, masks)
run["config"] = {"model": MODEL, "src": SRC, "tgt": TGT, "latent": LATENT,
                 "k_shot": 5, "prompt_seed": 0, "ctrl_seed": 0, "lens": "ln_final", "hook": "resid_post"}
run_name = f"{MODEL.split('/')[-1]}_{SRC}_{TGT}_{LATENT}"
print("saved", save_run(run, OUT / f"{run_name}.pt"))

curves, top1, correct = run["curves"], run["top1"], np.array(run["correct"])
print({name: tuple(c.shape) for name, c in curves.items()})

In [ ]:
# 1. The loop reproduces the single-prompt result from Step 3
for name in ["tgt_first", "lat_first", "rank_lat", "rank_lat_ctrl"]:
    print(name, torch.allclose(curves[name][7], m[name].cpu(), equal_nan=True))

# 2. Does the model actually translate? (final layer top-1 == target first token)
print(f"final-layer accuracy: {correct.mean():.2f}  ({correct.sum()}/{len(df)})")
print("wrong:", [(df.at[j, 'concept_id'], df.at[j, TGT], tok.decode(top1[j])) for j in np.where(~correct)[0][:10]])

# 3. Control collisions (rows with nan control)
print("nan control rows:", torch.isnan(curves["rank_lat_ctrl"][:, 0]).sum().item())

# 4. First look: median rank across prompts, every 3rd layer
med = {k: curves[k].nanmedian(dim=0).values for k in ["rank_tgt", "rank_lat", "rank_lat_ctrl"]}
for L in range(0, N_LAYERS, 3):
    print(f"L{L:2d}  median rank  tgt={med['rank_tgt'][L]:7.0f}  lat={med['rank_lat'][L]:7.0f}  ctrl={med['rank_lat_ctrl'][L]:7.0f}")

In [ ]:
# Script curves are probabilities of disjoint token sets → each ≤ 1, and their sum ≤ 1
tot = sum(curves[f"script_{s}"] for s in scripts)
print("script sum max:", tot.max().item())

# Early-layer target lead: is it specific to the concept, or "any Chinese word"?
med = {k: curves[k][correct].nanmedian(dim=0).values
       for k in ["rank_tgt", "rank_tgt_ctrl", "rank_lat", "rank_lat_ctrl"]}
print(f"correct-only prompts: {correct.sum()}")
for L in range(0, N_LAYERS, 3):
    print(f"L{L:2d}  tgt={med['rank_tgt'][L]:6.0f} tgt_ctrl={med['rank_tgt_ctrl'][L]:6.0f}  |  "
          f"lat={med['rank_lat'][L]:6.0f} lat_ctrl={med['rank_lat_ctrl'][L]:6.0f}  |  "
          + "  ".join(f"{s}={curves[f'script_{s}'][:, L].mean():.2f}" for s in scripts))

## Step 5 · Plots (`plotting.plot_run`, `plotting.plot_paired`)

Loads from the saved file, so this section runs without the model. Color = role (blue target, orange latent),
solid = matched word, dashed = control word.

In [ ]:
run = load_run(OUT / f"{run_name}.pt")
for measure_name in ["first", "prefix"]:
    fig = plot_run(run, measure=measure_name)
    fig.savefig(OUT / f"{run_name}_{measure_name}.png", dpi=150, bbox_inches="tight")

Paired comparison: matched − control **per prompt**, then mean ± 95% CI. Removes prompt-to-prompt variation;
above 0 = concept-specific signal. This is the statistic C1 will use (switch − matched non-switch).

In [ ]:
fig = plot_paired(run, measure="first")
fig.savefig(OUT / f"{run_name}_paired.png", dpi=150, bbox_inches="tight")